# Topic 27 — Neural Network Fundamentals
### Theory → activation functions → a 2-layer network from scratch (forward + backprop) → loss functions.

A neural network is a stack of **layers**, each made of **neurons**. Each neuron computes a
weighted sum of its inputs plus a bias, then passes that through an **activation function**.
Stacking layers lets the network learn increasingly complex, non-linear patterns.

```text
Input layer -> Hidden layer(s) -> Output layer
     x       ->   z = Wx+b, a=activation(z)  ->  prediction
```

Training = **forward propagation** (compute a prediction) -> **loss** (how wrong was it) ->
**backpropagation** (compute how much each weight contributed to the error, via the chain rule
from Topic 4) -> **gradient descent** (nudge every weight to reduce the loss). Repeat.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## 1. Activation functions

Without a non-linear activation, stacking layers would collapse into just one big linear function
(no matter how many layers) — activations are what let networks learn curved, complex boundaries.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
def tanh(z): return np.tanh(z)
def relu(z): return np.maximum(0, z)
def leaky_relu(z, alpha=0.01): return np.where(z > 0, z, alpha * z)
def softmax(z):
    exp_z = np.exp(z - np.max(z))   # subtract max for numerical stability
    return exp_z / exp_z.sum()

z = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, fn) in zip(axes, [("sigmoid", sigmoid), ("tanh", tanh), ("relu", relu), ("leaky_relu", leaky_relu)]):
    ax.plot(z, fn(z))
    ax.set_title(name)
    ax.axhline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()

print("softmax example (turns any vector into a probability distribution that sums to 1):")
scores = np.array([2.0, 1.0, 0.1])
print(softmax(scores), " sum =", softmax(scores).sum())

**When to use which:**
- **Sigmoid**: output layer for binary classification (gives a 0-1 probability, same as Topic 8).
- **Tanh**: like sigmoid but centered at 0 (range -1 to 1) — was common in hidden layers, less so now.
- **ReLU**: the default choice for hidden layers in modern networks — fast, avoids some
  training problems sigmoid/tanh have with very deep networks.
- **Leaky ReLU**: a small fix for ReLU's "dying neuron" problem (a ReLU stuck outputting 0 forever).
- **Softmax**: output layer for MULTI-class classification — converts scores into probabilities that sum to 1.

## 2. Loss functions

In [ ]:
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def binary_cross_entropy(y_true, y_pred, eps=1e-12):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def categorical_cross_entropy(y_true_onehot, y_pred_probs, eps=1e-12):
    y_pred_probs = np.clip(y_pred_probs, eps, 1)
    return -np.mean(np.sum(y_true_onehot * np.log(y_pred_probs), axis=-1))

print("MSE:", mse_loss(np.array([1, 0]), np.array([0.9, 0.2])), " -> regression tasks")
print("Binary cross-entropy:", binary_cross_entropy(np.array([1, 0]), np.array([0.9, 0.2])), " -> 2-class classification")

y_true_oh = np.array([0, 1, 0])   # true class is index 1, one-hot encoded
y_pred_probs = np.array([0.1, 0.7, 0.2])
print("Categorical cross-entropy:", categorical_cross_entropy(y_true_oh, y_pred_probs), " -> multi-class classification")

## 3. Forward propagation — building a tiny 2-layer network by hand

Architecture: 2 input features -> 3 hidden neurons (ReLU) -> 1 output neuron (sigmoid), for
binary classification. This is exactly the "neuron/weights/bias/layers" vocabulary applied concretely.

In [ ]:
# Initialize weights and biases randomly (small values)
n_input, n_hidden, n_output = 2, 3, 1

W1 = rng.normal(0, 0.5, size=(n_input, n_hidden))    # input -> hidden weights
b1 = np.zeros(n_hidden)
W2 = rng.normal(0, 0.5, size=(n_hidden, n_output))   # hidden -> output weights
b2 = np.zeros(n_output)

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1        # linear combination, layer 1
    a1 = relu(z1)            # activation, layer 1 (hidden layer output)
    z2 = a1 @ W2 + b2       # linear combination, layer 2
    a2 = sigmoid(z2)         # activation, layer 2 (final prediction)
    cache = (X, z1, a1, z2, a2)   # save intermediate values -- needed for backprop
    return a2, cache

X_sample = np.array([[1.0, 2.0]])
prediction, cache = forward(X_sample, W1, b1, W2, b2)
print("prediction (before any training):", prediction)

## 4. Backpropagation — the chain rule, applied layer by layer

Backprop computes `d(loss)/d(each weight)` by working BACKWARD from the output, reusing
intermediate values via the chain rule (Topic 4, Part B4). This is genuinely the least intuitive
part of deep learning — go slowly here.

In [ ]:
def relu_derivative(z):
    return (z > 0).astype(float)

def backward(y_true, W1, b1, W2, b2, cache):
    X, z1, a1, z2, a2 = cache
    n = X.shape[0]

    # Output layer gradient: for sigmoid + binary cross-entropy, this combination
    # conveniently simplifies to just (prediction - true_label)
    dz2 = a2 - y_true                       # shape (n, 1)
    dW2 = a1.T @ dz2 / n
    db2 = np.sum(dz2, axis=0) / n

    # Hidden layer gradient: propagate the error backward through W2, then through ReLU's derivative
    da1 = dz2 @ W2.T
    dz1 = da1 * relu_derivative(z1)
    dW1 = X.T @ dz1 / n
    db1 = np.sum(dz1, axis=0) / n

    return dW1, db1, dW2, db2

y_true_sample = np.array([[1.0]])   # pretend the correct label for X_sample is class 1
dW1, db1, dW2, db2 = backward(y_true_sample, W1, b1, W2, b2, cache)
print("gradient shapes:", dW1.shape, db1.shape, dW2.shape, db2.shape)
print("dW2 (how much each hidden->output weight should change):\n", dW2)

## 5. Putting it together: a full training loop on a tiny dataset

In [ ]:
# XOR-like toy problem: NOT linearly separable (a single logistic regression, Topic 8, cannot solve this)
X_train = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
y_train = np.array([[0], [1], [1], [0]], dtype=float)   # XOR pattern

W1 = rng.normal(0, 1, size=(2, 4))
b1 = np.zeros(4)
W2 = rng.normal(0, 1, size=(4, 1))
b2 = np.zeros(1)

lr = 0.5
loss_history = []

for epoch in range(3000):
    pred, cache = forward(X_train, W1, b1, W2, b2)
    loss = binary_cross_entropy(y_train, pred)
    loss_history.append(loss)

    dW1, db1, dW2, db2 = backward(y_train, W1, b1, W2, b2, cache)
    W1 -= lr * dW1; b1 -= lr * db1
    W2 -= lr * dW2; b2 -= lr * db2

final_pred, _ = forward(X_train, W1, b1, W2, b2)
print("final predictions:\n", np.round(final_pred, 3))
print("true labels:\n", y_train.flatten())

plt.figure(figsize=(5, 4))
plt.plot(loss_history)
plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("Training loss on XOR (a single-layer model could NEVER solve this)")
plt.show()

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Change n_hidden to 2 (instead of 4 or 3) and re-run the XOR training loop -- does it still converge?
# 2. Swap relu/relu_derivative for tanh/its derivative (1 - tanh(z)**2) throughout forward/backward
#    and compare convergence speed.
# 3. Plot the final decision boundary of the trained XOR network by predicting over a grid of points
#    (reuse the meshgrid pattern from Topic 8's decision-boundary plot).
# 4. In one sentence: why couldn't Topic 8's plain logistic regression solve the XOR problem,
#    but this 2-layer network can?

---
### Next up: **Topic 28 — Deep Learning Optimization** (SGD, momentum, Adam, learning-rate scheduling).

Say "next" when you're ready.